# OR entry method: next candle open versus 15-minute OR-line limit

One **first completed close breakout** per ticker/day/timeframe; keep the 30-minute OR (09:30–09:59 ET), 1-, 2-, 5-minute confirmation definitions, no QQQ/ticker-gap/day filters, and Nov 4–5 exclusion. Both hypothetical entry plans receive the same preselected signal. No later signal replaces an unfilled limit order.

**Only entry method changes.** A next-open order fills at the immediately next one-minute candle open. An OR-line buy limit for an upper breakout, or sell-short limit for a lower breakout, waits for a touch of the broken boundary for up to 15 minutes after the signal. If the opening price of a later minute is already marketable, fill at that open; otherwise if its range touches the boundary, fill at the boundary. Never fill in the signal candle.

Both plans: stop 1% from the *actual filled entry*, target 2% from filled entry, 5 basis points adverse entry and exit slippage, 15:55 ET hard exit; timed exit at or below −0.5% is loss, other timed exits neutral. On a bar touching stop and target use stop first. If a limit fills within a candle via its wick, its stop may trigger that candle; its target is only eligible starting the **following** candle because one-minute OHLC does not establish that the target happened after the intrabar fill. At a marketable opening fill both are active immediately. Model borrow availability and real bid/ask separately.

Reports show signal count, filled and unfilled limits, wins/losses/neutrals for filled trades, R per fill **and per original signal** (assigning 0R to unfilled limits); by month/date, and same ticker-day paired results. Trades across the three candle sizes are alternative hypothetical plans, not three independent opportunities. One-minute completeness is required through exit to avoid silently scoring missing bars as unfilled or neutral.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd, numpy as np, re
from IPython.display import display
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
ROOT=next((p for p in (BASE/'Backtesting',BASE/'backtesting') if p.is_dir()),None)
if ROOT is None: raise FileNotFoundError('Backtesting folder missing in MyDrive')
OUT=BASE/'OR_Entry_Method_1m_2m_5m'; OUT.mkdir(exist_ok=True)
TZ='America/New_York'
OR_MIN=30
SIGNAL_LAST='15:00'
SESSION_END='15:55'
HORIZONS=(15,30,60,'session_end')
TIMEFRAMES=(1,2,5)
EXCLUDED={'2026-11-04','2026-11-05'}
paths={}
csv_candidates=list(ROOT.rglob('*_1m_*.csv'))
print('Data folder:',ROOT)
print('Candidate one-minute files:',len(csv_candidates))
print('Example paths:',[str(f.relative_to(ROOT)) for f in csv_candidates[:3]])
for f in csv_candidates:
    m=re.match(r'(.+)_1m_[0-9]{4}-[0-9]{2}-[0-9]{2}$',f.stem,re.I)
    if m: paths.setdefault(m.group(1).upper(),[]).append(f)
print('Tickers:',len(paths),'one-minute CSV files:',sum(map(len,paths.values())))
if not paths:
    raise FileNotFoundError('No matching CSVs. Check Data folder and Example paths above; reopen the latest notebook if running a saved Colab copy.')


In [ ]:
def sessions_for_ticker(file_list):
    parts={}
    for f in sorted(file_list):
        try:
            x=pd.read_csv(f)
            tc=next((c for c in x if c.lower() in ('datetime','timestamp','date')),x.columns[0])
            t=pd.to_datetime(x[tc],utc=True,errors='coerce').dt.tz_convert(TZ)
            y=pd.DataFrame({'time':t})
            for c in ('Open','High','Low','Close','Volume'):
                y[c]=pd.to_numeric(x[c],errors='coerce') if c in x else np.nan
            y=y.dropna(subset=['time','Open','High','Low','Close'])
            y=y.set_index('time').between_time('09:30','15:59')
            for date,g in y.groupby(y.index.strftime('%Y-%m-%d')):
                parts.setdefault(date,[]).append(g)
        except Exception as e: print('Skipped',f.name,str(e)[:90])
    out={}
    for date,groups in parts.items():
        d=pd.concat(groups).sort_index()
        d=d[~d.index.duplicated(keep='last')]
        out[date]=d
    return out

def event_candidates(d,date,minutes,high,low):
    start=pd.Timestamp(date+' 09:30',tz=TZ)
    if minutes==1: bars=d[['Open','High','Low','Close']].copy()
    else:
        bars=d[['Open','High','Low','Close']].resample(
            f'{minutes}min',origin=start,label='left',closed='left'
        ).agg({'Open':'first','High':'max','Low':'min','Close':'last'})
        count=d.Close.resample(f'{minutes}min',origin=start,label='left',closed='left').count()
        bars=bars[count.eq(minutes)].dropna()
    prior={'upper':False,'lower':False}
    events=[]
    for stamp,c in bars.iterrows():
        signal=stamp+pd.Timedelta(minutes=minutes)
        if signal<start+pd.Timedelta(minutes=OR_MIN): continue
        if signal>pd.Timestamp(date+' '+SIGNAL_LAST,tz=TZ):break
        upper=bool(c.Close>high);lower=bool(c.Close<low)
        if upper and not prior['upper']:
            events.append(dict(signal_time=signal,side='upper',boundary=high,signal_close=float(c.Close)))
        if lower and not prior['lower']:
            events.append(dict(signal_time=signal,side='lower',boundary=low,signal_close=float(c.Close)))
        prior={'upper':upper,'lower':lower}
    return events


In [ ]:
SLIPPAGE_BPS=5
STOP_FRAC=.01
TARGET_FRAC=.02
LIMIT_WAIT_MIN=15
METHODS=('next_candle_open','OR_line_limit')

def get_entry(d,e,method):
    signal=e['signal_time']
    cutoff=pd.Timestamp(signal.date().isoformat()+' '+SESSION_END,tz=TZ)
    if method=='next_candle_open':
        if signal not in d.index:return None,'missing_next_open'
        return (signal,float(d.loc[signal,'Open']),'opening'),None
    end=min(signal+pd.Timedelta(minutes=LIMIT_WAIT_MIN),cutoff)
    x=d.loc[(d.index>=signal)&(d.index<end)]
    side=e['side'];boundary=e['boundary']
    for t,c in x.iterrows():
        marketable=(c.Open<=boundary) if side=='upper' else (c.Open>=boundary)
        touches=(c.Low<=boundary) if side=='upper' else (c.High>=boundary)
        if marketable:return (t,float(c.Open),'opening'),None
        if touches:return (t,boundary,'intrabar'),None
    return None,'unfilled_15m'

def replay(d,date,e,fill):
    t,raw,fill_type=fill
    sign=1 if e['side']=='upper' else -1
    end=pd.Timestamp(date+' '+SESSION_END,tz=TZ)
    expected=pd.date_range(t,end,freq='min',inclusive='left')
    if not len(expected):return None
    x=d.reindex(expected)
    if x[['Open','High','Low','Close']].isna().any().any():return None
    entry=raw*(1+sign*SLIPPAGE_BPS/10000)
    stop=entry*(1-sign*STOP_FRAC);target=entry*(1+sign*TARGET_FRAC)
    exit_time=x.index[-1];exit_raw=float(x.iloc[-1].Close);outcome='time'
    for j,(q,c) in enumerate(x.iterrows()):
        bad=bool(c.Low<=stop) if sign==1 else bool(c.High>=stop)
        good=bool(c.High>=target) if sign==1 else bool(c.Low<=target)
        if bad:
            outcome='stop';exit_time=q;exit_raw=stop;break
        if good and (j>0 or fill_type=='opening'):
            outcome='target';exit_time=q;exit_raw=target;break
    exit_price=exit_raw*(1-sign*SLIPPAGE_BPS/10000)
    net_frac=sign*(exit_price-entry)/entry
    result='win' if outcome=='target' else 'loss' if outcome=='stop' or net_frac<=-.005 else 'neutral'
    return dict(fill_time=t,fill_type=fill_type,entry_price=entry,
       exit_time=exit_time,exit_price=exit_price,outcome=outcome,result=result,
       net_pct=100*net_frac,R=net_frac/STOP_FRAC)

In [ ]:
signal_rows=[];fill_rows=[];coverage=[]
for count,ticker in enumerate(sorted(paths),1):
    for date,d in sessions_for_ticker(paths[ticker]).items():
        if date in EXCLUDED:continue
        start=pd.Timestamp(date+' 09:30',tz=TZ)
        opening=d.reindex(pd.date_range(start,periods=OR_MIN,freq='min'))
        if opening[['Open','High','Low','Close']].isna().any().any():
            coverage.append(dict(ticker=ticker,date=date,reason='incomplete_OR'));continue
        high=float(opening.High.max());low=float(opening.Low.min())
        if high<=low:continue
        for tf in TIMEFRAMES:
            ev=event_candidates(d,date,tf,high,low)
            if not ev:continue
            e=ev[0]
            base=dict(ticker=ticker,date=date,timeframe_min=tf,side=e['side'],
                      signal_time=e['signal_time'],boundary=e['boundary'],
                      or_high=high,or_low=low)
            # Restrict to a common fully observed path before comparing methods.
            expected=pd.date_range(e['signal_time'],
                pd.Timestamp(date+' '+SESSION_END,tz=TZ),freq='min',inclusive='left')
            if not len(expected) or d.reindex(expected)[['Open','High','Low','Close']].isna().any().any():
                coverage.append({**base,'reason':'incomplete_post_signal_path'});continue
            for method in METHODS:
                fill,reason=get_entry(d,e,method)
                row={**base,'method':method,'status':reason or 'filled'}
                if fill is not None:
                    result=replay(d,date,e,fill)
                    if result is None:
                        row['status']='incomplete_fill_path'
                    else:
                        fill_rows.append({**row,**result})
                signal_rows.append(row)
    if count%100==0:print('Loaded',count,'/',len(paths),'tickers')
signals=pd.DataFrame(signal_rows);fills=pd.DataFrame(fill_rows)
if signals.empty or fills.empty:raise RuntimeError('No fully observed signals/fills; inspect coverage')
assert signals.groupby(['ticker','date','timeframe_min']).method.nunique().eq(2).all()
signals.to_csv(OUT/'paired_signal_audit.csv',index=False)
fills.to_csv(OUT/'entry_method_fills.csv',index=False)
pd.DataFrame(coverage).to_csv(OUT/'coverage_audit.csv',index=False)
print('Unique first signals:',signals[['ticker','date','timeframe_min']].drop_duplicates().shape[0],
    '| hypothetical fills:',len(fills))
print('Status counts:');display(signals.groupby(['timeframe_min','side','method','status']).size().reset_index(name='signals'))

In [ ]:
def summarize(g):
    base=signals.loc[g.index]
    n=len(base);matched=fills.merge(base[['ticker','date','timeframe_min','method']],
      on=['ticker','date','timeframe_min','method'],how='inner')
    return pd.Series(dict(signals=n,filled=len(matched),unfilled=n-len(matched),
      fill_pct=100*len(matched)/n if n else np.nan,
      wins=int(matched.result.eq('win').sum()),losses=int(matched.result.eq('loss').sum()),
      neutrals=int(matched.result.eq('neutral').sum()),
      win_pct_of_fills=100*matched.result.eq('win').mean() if len(matched) else np.nan,
      net_R=matched.R.sum(),mean_R_per_fill=matched.R.mean(),
      mean_R_per_signal=matched.R.sum()/n if n else np.nan,
      tickers=base.ticker.nunique(),dates=base.date.nunique()))
summary=signals.groupby(['timeframe_min','side','method'],dropna=False).apply(
  summarize,include_groups=False).reset_index()
assert (summary.wins+summary.losses+summary.neutrals==summary.filled).all()
summary.to_csv(OUT/'entry_method_summary.csv',index=False)
display(summary.round(3))
# Month and date splits, using the exact same per-method signal population.
signals['month']=signals.date.str[:7]
for name,group in [('month',['month','timeframe_min','side','method']),
                   ('date',['date','timeframe_min','side','method'])]:
    out=signals.groupby(group,dropna=False).apply(summarize,include_groups=False).reset_index()
    out.to_csv(OUT/('entry_method_by_'+name+'.csv'),index=False)
    if name=='month':display(out.round(3))
print('Saved output folder',OUT)

## Interpretation

The comparison is **per original signal**: an unfilled OR-line limit has no trade and contributes 0R. Reported win percentages use *filled* orders only and must be read with the fill rate. A limit that fills at an opening price inside the OR is marked `opening`; a boundary-touch fill is marked `intrabar`. Inspect these separately before any live use.

The stop and target are percentages of each method's own fill price, so changing entry naturally changes both prices while their percentage settings stay fixed. Compare upper and lower directions separately, then month/date stability, and do not select a method solely on pooled R. The previously studied opening-volume top-third subset is a different filtered population and should not be conflated with this unfiltered first-signal study.